In [17]:
!pip install chardet rapidfuzz beautifulsoup4 lxml tqdm openpyxl

In [1]:
from pathlib import Path
import re, shutil, chardet
from tqdm import tqdm
from bs4 import BeautifulSoup
import unicodedata as ud
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter


In [19]:
# ---------- 讀取關鍵字 ----------
def load_keywords(path: Path):
    kws = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            kw = line.strip()
            if kw:
                kws.append(kw)
    return kws


# ---------- 讀檔（多編碼容錯） ----------
def read_text_best_effort(path: Path, max_bytes: int | None = None) -> str:
    raw = path.read_bytes() if max_bytes is None else path.read_bytes()[:max_bytes]
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        pass
    enc = (chardet.detect(raw).get("encoding") or "latin-1")
    try:
        return raw.decode(enc, errors="replace")
    except Exception:
        return raw.decode("latin-1", errors="replace")


# ---------- HTML → 純文字 ----------
def html_to_text(src_html: str) -> str:
    soup = BeautifulSoup(src_html, "lxml")
    return soup.get_text(separator=" ")


# ---------- 人眼式正規化 ----------
_ZW_CHARS = r"\u200b\u200c\u200d\u2060"
_MISC_CTRL = r"\ufeff\u00ad"
_UNICODE_SPACES = r"\u00a0\u1680\u180e\u2000-\u200a\u202f\u205f\u3000\u2028\u2029"

def normalize_text(s: str, *, remove_all_spaces: bool = True) -> str:
    s = ud.normalize("NFKC", s)
    s = "".join(ch for ch in ud.normalize("NFKD", s) if ud.category(ch) != "Mn")
    s = s.lower()
    s = re.sub(r"-\s*\n\s*", "", s)
    s = s.replace("\r", "\n").replace("\t", " ")
    s = re.sub(r"\s*\n\s*", " ", s)
    s = re.sub(f"[{_ZW_CHARS}{_MISC_CTRL}{_UNICODE_SPACES}]", " ", s)
    kept = [ch for ch in s if ch.isspace() or ud.category(ch).startswith(("L","N"))]
    s = "".join(kept)
    s = re.sub(r"\s+", " ", s).strip()
    if remove_all_spaces:
        s = s.replace(" ", "")
    return s


In [21]:
# ---------- 關鍵字比對（整詞精確，比對次數 + 不同關鍵字清單） ----------
def find_keywords_stats(text: str, compiled_patterns):
    total_occurrences = 0
    distinct_hits = set()

    for kw, pattern in compiled_patterns:
        # 計算該關鍵字在全文中出現幾次（整詞）
        cnt = sum(1 for _ in pattern.finditer(text))
        if cnt > 0:
            total_occurrences += cnt
            distinct_hits.add(kw)  # 用原始關鍵字字面回報

    return total_occurrences, distinct_hits



def classify_file(p: Path, compiled_patterns):
    try:
        raw_html = read_text_best_effort(p)
        text = html_to_text(raw_html)
        # 保留空白，才能做整詞比對（\b 邊界要靠空白/字界）
        norm_text = normalize_text(text, remove_all_spaces=False)

        total_occurrences, distinct_hits = find_keywords_stats(norm_text, compiled_patterns)

        if total_occurrences > 0:
            hits_list = sorted(distinct_hits)  # 唯一清單（可依需要排序/不排序）
            return {
                "file_name": p.name,
                "file_path": p,  # <-- ★★★ 新增：回傳完整路徑，以便複製 ★★★
                "total_keyword_occurrences": total_occurrences,   # ① 總出現次數（含重複）
                "num_distinct_keywords": len(distinct_hits),      # ② 不同關鍵字數
                "keywords_found": hits_list,                      # ③ 哪些關鍵字（唯一清單）
            }
        else:
            # 沒找到關鍵字 → 不回傳（用 None 表示略過）
            return None

    except Exception as e:
        tqdm.write(f"[ERROR] {p}: {e}")
        return None

In [33]:
# ---------- 關鍵字預先編譯（整詞精確比對用） ----------
def compile_keyword_patterns(keywords_norm):
    compiled = []
    for kw in keywords_norm:
        norm_kw = normalize_text(str(kw), remove_all_spaces=False).strip()
        if not norm_kw:
            continue
        tokens = norm_kw.split()
        pat = re.compile(r"\b" + r"\s+".join(map(re.escape, tokens)) + r"\b")
        compiled.append((kw, pat))
    return compiled
    

In [41]:

# ---------- 輔助函數：將大清單切分為小批次 ----------
def chunked_list(lst, n):
    """將 lst 列表切分為每份 n 個的小列表"""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# ---------- 核心 main 函數 (已優化時間複雜度) ----------
def main(batch_to_run: int, processing_chunk_size: int = 100000):
    
    if batch_to_run < 1:
        print("錯誤：批次編號 (batch_to_run) 必須大於等於 1")
        return pd.DataFrame()

    total_start_time = perf_counter()
    
    # === 參數設定 ===
    KW_FILE = Path("/Users/wanghao/Downloads/Internal_Control/Audit_Term_keyword_Wu_revision_03202025.txt")
    
    # === 核心修改：讀取預先掃描好的清單 ===
    LIST_FILE_INPUT = Path("/Volumes/One Touch/8k_has_keyword_file_html/all_files_list.txt")
    
    OUTPUT_ROOT_BASE = Path("/Volumes/One Touch/8k_has_keyword_file_html")
    BATCH_OUTPUT_ROOT = OUTPUT_ROOT_BASE / f"Processing_Batch_{batch_to_run}"
    BATCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    
    EXCEL_OUTPUT_FILE = BATCH_OUTPUT_ROOT / f"batch_{batch_to_run}_summary.xlsx"
    COPY_BATCH_SIZE = 5000
    
    # === 讀關鍵字 ===
    print(f"載入關鍵字並編譯正規表達式...")
    KEYWORDS = load_keywords(KW_FILE)
    compiled_patterns = compile_keyword_patterns(KEYWORDS)

    # === 核心修改：從文字檔讀取檔案清單 ===
    print(f"正在從 {LIST_FILE_INPUT} 讀取檔案清單...")
    try:
        all_files = []
        with open(LIST_FILE_INPUT, "r", encoding="utf-8") as f:
            for line in f:
                stripped_line = line.strip()
                if stripped_line:
                    all_files.append(Path(stripped_line)) # 轉回 Path 物件
    except FileNotFoundError:
        print(f"錯誤：找不到檔案清單 {LIST_FILE_INPUT}")
        print("   請先執行 步驟 1：建立檔案索引！")
        return pd.DataFrame()
        
    total_files_count = len(all_files)
    print(f"從清單讀取 {total_files_count} 筆檔案路徑")

    # === 計算並選取指定批次的檔案 ===
    file_chunks = list(chunked_list(all_files, processing_chunk_size))
    num_chunks = len(file_chunks)
    
    batch_index_to_run = batch_to_run - 1 # 批次 1 對應索引 0
    
    if batch_index_to_run >= num_chunks:
        print(f"錯誤：您想處理第 {batch_to_run} 批，但檔案總共只被分為 {num_chunks} 批。")
        return pd.DataFrame()

    batch_files_to_process = file_chunks[batch_index_to_run]
    
    print("-" * 40)
    print(f"準備處理第 {batch_to_run}/{num_chunks} 批次")
    print(f"   (檔案索引 {batch_index_to_run * processing_chunk_size} 到 {min((batch_index_to_run + 1) * processing_chunk_size, total_files_count) - 1})")
    print(f"   本批次共 {len(batch_files_to_process)} 筆檔案。")
    print(f"   結果將存放在: {BATCH_OUTPUT_ROOT}")
    print("-" * 40)

    # === 多執行緒跑分類 (僅針對當前批次) ===
    batch_results = []
    copied_file_count = 0 # 本批次的複製計數器

    with ThreadPoolExecutor() as exe:
        futures = [exe.submit(classify_file, p, compiled_patterns) for p in batch_files_to_process]
        
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"批次 {batch_to_run} 分類複製中", unit="file"):
            res = fut.result()
            if res is not None:
                batch_results.append(res) 

                # --- 執行複製與分批 (使用 5000 筆的設定) ---
                source_path = res["file_path"]
                subfolder_index = copied_file_count // COPY_BATCH_SIZE
                subfolder_name = f"copy_batch_{subfolder_index + 1}" # (原 5000 筆的邏輯)
                dest_folder = BATCH_OUTPUT_ROOT / subfolder_name
                dest_folder.mkdir(parents=True, exist_ok=True)
                
                try:
                    dest_path = dest_folder / source_path.name
                    shutil.copy2(source_path, dest_path)
                    copied_file_count += 1
                except shutil.SameFileError:
                    pass 
                except Exception as e:
                    tqdm.write(f"[COPY ERROR] 複製 {source_path.name} 失敗: {e}")
    
    # --- 當前批次處理完畢 ---
    if not batch_results:
        print(f"第 {batch_to_run} 批次處理完畢，但沒有任何檔案命中關鍵字。")
        return pd.DataFrame()

    # === 彙整與排序 ===
    df = pd.DataFrame(batch_results, columns=["file_name", "total_keyword_occurrences", "num_distinct_keywords", "keywords_found", "file_path"])
    df.sort_values(by="total_keyword_occurrences", ascending=False, inplace=True)
    df_for_excel = df.drop(columns=["file_path"])

    total_elapsed = perf_counter() - total_start_time

    # === 總結報告 ===
    print(f"第 {batch_to_run} 批次處理完畢！")
    print(f"   共處理 {len(batch_files_to_process)} 檔案 → 有關鍵字 {len(df)} 檔")
    print(f"   成功複製 {copied_file_count} 個檔案至 {BATCH_OUTPUT_ROOT}")
    print(f"   總耗時：{total_elapsed:.2f} 秒")

    # === 儲存 Excel ===
    try:
        print(f"正在儲存 Excel 報表至 {EXCEL_OUTPUT_FILE}...")
        df_for_excel.to_excel(EXCEL_OUTPUT_FILE, index=False, engine="openpyxl")
        print("Excel 報表儲存完畢。")
    except Exception as e:
        print(f"儲存 Excel 失敗: {e}")

    return df

In [43]:
# 執行第 1 批 (5000 筆)
df_batch_1 = main(batch_to_run=1, processing_chunk_size=5000)
df_batch_1.head(10) # 顯示第 1 批的結果

KeyboardInterrupt: 

In [ ]:
# --- 獨立任務：產生檔案總清單 (一勞永逸) ---
import pandas as pd
from pathlib import Path
from time import perf_counter

start_scan = perf_counter()

INPUT_ROOT = Path("/Volumes/One Touch/8k_file_html")
LIST_FILE_OUTPUT = Path("/Volumes/One Touch/8k_has_keyword_file_html/all_files_list.txt")
LIST_FILE_OUTPUT.parent.mkdir(parents=True, exist_ok=True) # 確保目錄存在

all_files = sorted(INPUT_ROOT.rglob("*.html"))

try:
    with open(LIST_FILE_OUTPUT, "w", encoding="utf-8") as f:
        for file_path in all_files:
            f.write(str(file_path) + "\n")
    
    elapsed_scan = perf_counter() - start_scan
    print(f"檔案清單已儲存完畢！共 {len(all_files)} 筆檔案。")
    print(f"   儲存至: {LIST_FILE_OUTPUT}")
    print(f"   耗時: {elapsed_scan:.2f} 秒")

except Exception as e:
    print(f"儲存檔案清單失敗: {e}")

正在搜尋所有檔案，這會需要很長時間，請耐心等待...
